## Loading Features


In [2]:
from pathlib import Path
import json
import joblib
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score
)

PROJECT_DIR = Path.cwd()
ARTIFACTS = PROJECT_DIR / "artifacts"
MODELS = PROJECT_DIR / "models"
REPORTS = PROJECT_DIR / "reports"
REPORTS.mkdir(exist_ok=True)

X_train = joblib.load(ARTIFACTS / "05_X_train.joblib")
X_validation = joblib.load(ARTIFACTS / "05_X_validation.joblib")
X_test = joblib.load(ARTIFACTS / "05_X_test.joblib")

y_train = pd.read_csv(ARTIFACTS / "05_y_train.csv").squeeze("columns")
y_validation = pd.read_csv(ARTIFACTS / "05_y_validation.csv").squeeze("columns")
y_test = pd.read_csv(ARTIFACTS / "05_y_test.csv").squeeze("columns")

## Baseline

In [3]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_validation)

baseline_f1 = f1_score(
    y_validation,
    baseline_pred,
    zero_division=0
)

print("Baseline F1:", baseline_f1)
print(classification_report(y_validation, baseline_pred, zero_division=0))

Baseline F1: 0.0
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     17730
           1       0.00      0.00      0.00      1565

    accuracy                           0.92     19295
   macro avg       0.46      0.50      0.48     19295
weighted avg       0.84      0.92      0.88     19295



##  Training a Baseline/Preliminary Model

In [4]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)
validation_pred = model.predict(X_validation)
validation_prob = model.predict_proba(X_validation)[:, 1]

print(classification_report(y_validation, validation_pred, zero_division=0))
print("Validation F1:", f1_score(y_validation, validation_pred, zero_division=0))
print("Validation ROC-AUC:", roc_auc_score(y_validation, validation_prob))
print("Validation PR-AUC:", average_precision_score(y_validation, validation_prob))

              precision    recall  f1-score   support

           0       0.94      0.69      0.79     17730
           1       0.13      0.51      0.20      1565

    accuracy                           0.67     19295
   macro avg       0.53      0.60      0.50     19295
weighted avg       0.88      0.67      0.75     19295

Validation F1: 0.20254440105806776
Validation ROC-AUC: 0.6493986474432786
Validation PR-AUC: 0.13305803602954555


## Model Tuning


In [5]:
param_grid = {
    "C": [0.01, 0.1, 1, 10]
}

grid = GridSearchCV(
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1
)

grid.fit(X_train, y_train)
print("Best parameters:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

Best parameters: {'C': 0.1}
Best CV F1: 0.20340075714103223


## Final Evaluation on Test Set


In [6]:
best_model = grid.best_estimator_
best_model.fit(X_train, y_train)

test_pred = best_model.predict(X_test)
test_prob = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, test_pred, zero_division=0))
print("Test F1:", f1_score(y_test, test_pred, zero_division=0))
print("Test Precision:", precision_score(y_test, test_pred, zero_division=0))
print("Test Recall:", recall_score(y_test, test_pred, zero_division=0))
print("Test ROC-AUC:", roc_auc_score(y_test, test_prob))
print("Test PR-AUC:", average_precision_score(y_test, test_prob))
print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))

              precision    recall  f1-score   support

           0       0.94      0.68      0.79     17731
           1       0.13      0.52      0.20      1565

    accuracy                           0.67     19296
   macro avg       0.53      0.60      0.50     19296
weighted avg       0.88      0.67      0.74     19296

Test F1: 0.20246913580246914
Test Precision: 0.12547819433817903
Test Recall: 0.5239616613418531
Test ROC-AUC: 0.6395463947098662
Test PR-AUC: 0.1284262809289564
Confusion matrix:
[[12016  5715]
 [  745   820]]


## Saving Model and Results

In [7]:
joblib.dump(best_model, MODELS / "final_model.joblib")

results = {
    "baseline_validation_f1": float(baseline_f1),
    "model_validation_f1": float(
        f1_score(y_validation, validation_pred, zero_division=0)
    ),
    "test_f1": float(f1_score(y_test, test_pred, zero_division=0)),
    "test_precision": float(
        precision_score(y_test, test_pred, zero_division=0)
    ),
    "test_recall": float(
        recall_score(y_test, test_pred, zero_division=0)
    ),
    "test_roc_auc": float(roc_auc_score(y_test, test_prob)),
    "test_pr_auc": float(average_precision_score(y_test, test_prob)),
    "best_parameters": grid.best_params_
}

with open(REPORTS / "results_summary.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("Saved final model and results summary")

Saved final model and results summary
